# sEEG preprocessing template
Author: Chang D. Liu

Credit: Andrey Zyryanov for bipolar reference 

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import mne
import numpy as np
import librosa
import os
import mat73
import re
import pandas as pd

# global variables
trigger_dur = 0.35
margin = 2 # margin at cropping
audio_channel = 'DC2'

subject = "TUE24"
path = f"../raw/{subject}_KEC.mat"
bad_prepend = 10 # in seconds, mark this much as bad before bad segments
bad_append = 5 # in seconds, mark this much as bad after bad segments

In [ ]:
data = mat73.loadmat(path)['data']
data.keys()

In [ ]:
data['label'] = np.array(data['label']).squeeze().tolist()

In [ ]:
# these are the typical labels of non-eeg channels
# add any other misc channels here if needed to correctly label eeg vs. misc channels in the Mne.Raw object
misc = ['EKG1', 'EKG 2', 'DC1', 'DC2', 'DC3', 'DC4', 'DC5', 'DC6', 'DC7', 'DC8', 'DC9', 'DC10', 'DC11', 'DC12', 'DC13', 'DC14', 'DC15', 'DC16', 'TRIG', 'OSAT', 'PR', 'Pleth'] + \
       ['c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'c10', 'c11', 'c12', 'c13', 'c14', 'c15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C27', 'EOG1', 'EOG2', 'EKG 1', 'EKG 2', 'C32', 'c48',
        'c64', 'c80', 'c93', 'c94', 'C95', 'C96', 'C109', 'C110', 'C111', 'C112', 'C125', 'C126', 'C127', 'C128']

eeg = ['Fp1', 'Fp2', 'F7.', 'F8.', 'T3', 'T4', 'T5', 'T6', 'O1', 'O2', 'F3.', 'F4.', 'C3.', 'C4.', 'P3', 'P4', 'T1', 'T2', 'Fz', 'Cz', 'Pz', 'EOG1', 'EOG2', 'P6']

# defining channel types
ch_types = ['misc' if ch in misc else 'eeg' if ch in eeg else 'seeg' for ch in data['label']]

assert len(data['label']) == len(ch_types)

In [ ]:
# filter out zero channels
# Find channels where all values are 0
zero_channels = np.all(np.diff(data['trial'][0][:, :100]) == 0, axis=1)

# Filter out these channels
filtered_trial = data['trial'][0][~zero_channels]
print(f"Filtered trial shape: {filtered_trial.shape}")

# Filter out corresponding labels
filtered_labels = [label for label, is_zero in zip(data['label'], ~zero_channels) if is_zero]
print(f"Filtered labels shape: {len(filtered_labels)}")

# Channel types
filtered_ch_types = [ch_type for ch_type, is_zero in zip(ch_types, ~zero_channels) if is_zero]
print(f"Filtered channel types shape: {len(filtered_ch_types)}")

In [ ]:
# converting data into mne.Raw object
temp_info = mne.create_info(filtered_labels, # channel names
                            data['fsample'], # sampling frequency
                            ch_types=filtered_ch_types)

raw = mne.io.RawArray(filtered_trial/1000000, # converting to volts
                      temp_info,
                      first_samp=0,
                      copy='both')

raw.plot(duration = 15,
         clipping = None,
         highpass = 1,
         lowpass = None,
         scalings = 'auto')

In [ ]:
raw.plot_psd()

In [ ]:
# Find channels where all values are 0
zero_channels = np.all(np.diff(data['trial'][0][:, int(data['fsample'])*10:int(data['fsample'])*20]) == 0, axis=1)

# Filter out these channels
filtered_trial2 = data['trial'][0][~zero_channels]
print(f"Filtered trial shape: {filtered_trial2.shape}")

In [ ]:
zero_channels.shape

In [ ]:
np.array(data['label'])[~zero_channels]

Preliminary reference

In [ ]:
audio = raw.get_data(picks=[audio_channel]).squeeze()  # Extract audio channel data
audio

In [ ]:
# applying notch
raw.notch_filter(np.array((50,100,150)), n_jobs = 8)

# automatically prepare re-ref. scheme here
ch_names = np.array(raw.info['ch_names'])[np.array(raw.get_channel_types()) == 'seeg']
ch_groups = list(set([re.sub('[0-9]','',ch) for ch in ch_names]))

cathodes = []
anodes = []

for group in ch_groups:
    ch_subset = [ch for ch in ch_names if re.match(f'{group}[0-9]+?', ch) != None and ch not in raw.info['bads']]
    if ch_subset != []:
        anodes += ch_subset[1:len(ch_subset)]
        cathodes += ch_subset[0:len(ch_subset)-1]

bipol_scheme = pd.DataFrame({'anodes': anodes,
                             'cathodes': cathodes})

bipol_scheme['bip_ch_names'] = bipol_scheme['anodes'] + '-' + bipol_scheme['cathodes']

print('Suggested re-reference scheme:\n\n', bipol_scheme.to_string())

In [ ]:
# applying the reference
raw_bipol = mne.set_bipolar_reference(raw, list(bipol_scheme['anodes']), list(bipol_scheme['cathodes']), ch_name=list(bipol_scheme['bip_ch_names']),
                                      drop_refs=True, copy=True, verbose=None)

# keeping only the new bipolar channels
raw_bipol.pick(list(bipol_scheme['bip_ch_names']))

# examining data and PSD after notch and bipolar re-referencing
raw_bipol.plot(duration = 15,
               n_channels = len(raw_bipol.ch_names),
               clipping = None,
               highpass = None,
               lowpass = None)

raw_bipol.compute_psd().plot()

In [ ]:
assert False, "stop here to identify bad channels"

In [ ]:
# manually identify bad channels
bads = ["G7"]
raw.info['bads'] = bads

# cross-correlation

## using the whole audio (Preliminary check: works when there is no pause)

In [ ]:
from scipy import signal

# cross-correlation
def sort_audios(audios):
    pattern = r'(?:new)?part(\d+)_'
    return sorted(audios, key=lambda x: int(re.search(pattern, x).group(1)))
raw_sfreq = raw.info['sfreq']
audios = [f for f in os.listdir('../mono/') if f.endswith('.wav')]
audios = sort_audios(audios)
audio_onsets = []
audio_offsets = []
stimuli = []
for audio_file in audios:
    stim, _ = librosa.load('../mono/' + audio_file, sr=raw_sfreq)
    stimuli.append(stim)
    stim_rev = stim[::-1]
    # cross-correlating stimulus and audio channel, identifying onset
    correlation = signal.correlate(audio, stim, mode="full")
    lags = signal.correlation_lags(len(audio), len(stim), mode="full")
    assert correlation.shape[0] == lags.shape[0]
    if audio_file == 'part1_d3p6_wt.wav':
        while lags[np.argmax(correlation)] > len(audio)/2:
            correlation[np.argmax(correlation)] = -np.inf  # to avoid picking the wrong peak
        audio_onsets.append(lags[np.argmax(correlation)])  # manually adjust for part1_d3p6_wt.wav
    else:
        audio_onsets.append(lags[np.argmax(correlation)])
    # cross-correlating for offset
    correlation_rev = signal.correlate(audio[::-1], stim_rev, mode="full")
    lags_rev = signal.correlation_lags(len(audio[::-1]), len(stim_rev), mode="full")
    assert correlation_rev.shape[0] == lags_rev.shape[0]
    if audio_file == 'part1_d3p6_wt.wav':
        while lags_rev[np.argmax(correlation_rev)] < len(audio)/2:
            correlation_rev[np.argmax(correlation_rev)] = -np.inf  # to avoid picking the wrong peak
        audio_offsets.append(len(audio) - (lags_rev[np.argmax(correlation_rev)]))  # manually adjust for part1_d3p6_wt.wav
    else:
        audio_offsets.append(len(audio) - lags_rev[np.argmax(correlation_rev)])
    print(f"Processed {audio_file}: onset at {audio_onsets[-1]/raw_sfreq}, offset at {audio_offsets[-1]/raw_sfreq}")
    assert audio_offsets[-1] - audio_onsets[-1] == len(stim)
    assert audio[audio_onsets[-1]:audio_offsets[-1]].shape[0] == stim.shape[0]


In [ ]:
audio_s = np.arange(len(audio)) / raw_sfreq  # time axis in seconds

plt.figure(figsize=(10, 5))
plt.plot(audio_s, audio, label="Audio Sequence", alpha=0.7)

# Highlight parts between audio onsets and offsets
for onset, offset in zip(audio_onsets, audio_offsets):
    plt.axvspan(onset/raw_sfreq, offset/raw_sfreq, color='orange', alpha=0.3, label="Highlighted Segment")

# Add labels and legend
plt.title("Audio Sequence with Highlighted Segments")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(["Audio Sequence", "Highlighted Segments"], loc="upper right")
plt.show()

## cross-correlating the audio onset and offset

run this block if the audio playback is interrupted

In [ ]:
assert False, "stop here to modify interruption criteria"

In [ ]:
# center the audio at 0
# manually input a silence period based on the plots above

silence_range = (0, int(30*raw_sfreq)) # format: (start_sample, end_sample)
audio_centered = audio - np.mean(audio[silence_range[0]:silence_range[1]])

raw_sfreq = raw.info['sfreq']

In [ ]:
from scipy import signal

# manually input the plausible time ranges of each part based on the plots above
plausible_ranges = {
    'part1_d3p6_wt.wav': (0, 1300*raw_sfreq), # format: (start_sample, end_sample)
    'part2_d11p6_wt.wav': (1000*raw_sfreq, 2300*raw_sfreq),
    'part3_d10p5_wt.wav': (2000*raw_sfreq, 2800*raw_sfreq),
    'newpart4_d9p21_wt.wav': (2700*raw_sfreq, len(audio))
}

# parameters for detection
detection_onset = 30  # cross correlate the first n seconds of the stimulus for onset detection
detection_offset = 30  # cross correlate the last n seconds of the stimulus for offset detection

# cross-correlation
def sort_audios(audios):
    pattern = r'(?:new)?part(\d+)_'
    return sorted(audios, key=lambda x: int(re.search(pattern, x).group(1)))
raw_sfreq = raw.info['sfreq']
audios = [f for f in os.listdir('../mono/') if f.endswith('.wav')]
audios = sort_audios(audios)
audio_onsets_prelim = []
audio_offsets_prelim = []
stimuli = []
for audio_file in audios:
    stim, _ = librosa.load('../mono/' + audio_file, sr=raw_sfreq)
    stimuli.append(stim)
    stim_rev = stim[::-1]

    # audio onset and offset detection using the first and last n seconds of the stimulus
    stim_begin = stim[:int(detection_onset*raw_sfreq)]  # first n seconds
    stim_end = stim[-int(detection_offset*raw_sfreq):]  # last n seconds

    # using stim_begin for onset
    # cross-correlating stimulus and audio channel, identifying onset
    correlation = signal.correlate(audio, stim_begin, mode="full")
    lags = signal.correlation_lags(len(audio), len(stim_begin), mode="full")
    assert correlation.shape[0] == lags.shape[0]

    # apply plausible range constraints
    min_lag, max_lag = plausible_ranges[audio_file]
    while lags[np.argmax(correlation)] < min_lag or lags[np.argmax(correlation)] > max_lag:
        correlation[np.argmax(correlation)] = -np.inf  # to avoid picking the wrong peak
    audio_onsets_prelim.append(lags[np.argmax(correlation)])

    # using stim_end for offset
    # cross-correlating for offset
    correlation_rev = signal.correlate(audio[::-1], stim_end[::-1], mode="full")
    lags_rev = signal.correlation_lags(len(audio[::-1]), len(stim_end[::-1]), mode="full")
    assert correlation_rev.shape[0] == lags_rev.shape[0]

    # apply plausible range constraints
    sample_idx = len(audio) - lags_rev[np.argmax(correlation_rev)]
    while sample_idx < min_lag or sample_idx > max_lag or sample_idx - audio_onsets_prelim[-1] < len(stim):
        correlation_rev[np.argmax(correlation_rev)] = -np.inf  # to avoid picking the wrong peak
        sample_idx = len(audio) - lags_rev[np.argmax(correlation_rev)]
    audio_offsets_prelim.append(len(audio) - lags_rev[np.argmax(correlation_rev)])

    print(f"Processed {audio_file}: onset at {audio_onsets_prelim[-1]/raw_sfreq}, offset at {audio_offsets_prelim[-1]/raw_sfreq}")
    assert audio_offsets_prelim[-1] - audio_onsets_prelim[-1] >= len(stim)
    #assert audio[audio_onsets[-1]:audio_offsets[-1]].shape[0] == stim.shape[0]

# plotting
audio_s = np.arange(len(audio)) / raw_sfreq  # time axis in seconds
plt.figure(figsize=(10, 5))
plt.plot(audio_s, audio, label="Audio Sequence", alpha=0.7)

# Highlight parts between audio onsets and offsets
for onset, offset in zip(audio_onsets_prelim, audio_offsets_prelim):
    plt.axvspan(onset/raw_sfreq, offset/raw_sfreq, color='orange', alpha=0.3, label="Highlighted Segment")

# Add labels and legend
plt.title("Audio Sequence with Highlighted Segments")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(["Audio Sequence", "Aligned Segments"], loc="upper right")
plt.show()

## Identify interruptions

In [ ]:
# parameters for pause detection
amplitude_threshold = 0.002  # adjust based on audio amplitude scale
min_pause_duration_sec = 10  # minimum duration of pause in seconds

# pause detection
pauses = np.abs(audio_centered) < amplitude_threshold  # center around mean pause amplitude
pause_intervals = []
in_pause = False
min_pause_duration = int(min_pause_duration_sec * raw_sfreq)  # minimum pause duration of 5 seconds
for i, is_pause in enumerate(pauses):
    if is_pause and not in_pause:
        pause_start = i
        in_pause = True
    elif not is_pause and in_pause:
        pause_intervals.append((pause_start, i))
        in_pause = False

# Filter out short pauses
pause_intervals = [(start, end) for start, end in pause_intervals if (end - start) > min_pause_duration]
print(f'Number of pauses detected: {len(pause_intervals)}')

# Filter out pauses outside audio playback segments
interruptions_prelim = [(start, end) for start, end in pause_intervals if any(
    end < offset and start > onset for onset, offset in zip(audio_onsets_prelim, audio_offsets_prelim)
)]
print(f'Number of interruptions detected: {len(interruptions_prelim)}')

In [ ]:
# visualize interruptions and aligned segments
audio_s = np.arange(len(audio)) / raw_sfreq  # time axis in seconds
plt.figure(figsize=(10, 5))
plt.plot(audio_s, audio, label="Audio Sequence", alpha=0.7)

# Highlight parts between audio onsets and offsets
for onset, offset in zip(audio_onsets_prelim, audio_offsets_prelim):
    plt.axvspan(onset/raw_sfreq, offset/raw_sfreq, color='orange', alpha=0.3, label="Audio playback")

# Highlight interruptions
for i, (start, end) in enumerate(interruptions_prelim):
    plt.axvspan(start/raw_sfreq, end/raw_sfreq, color='red', linestyle='--', alpha=0.3, label = "Interruption")
    plt.text(int(np.mean([start/raw_sfreq, end/raw_sfreq])), 0, f"Interruption {i}", color='black', fontsize=8, ha='center')

# Add labels and legend
plt.title("Interruptions (red, 0-indexed)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(["Audio Sequence", "Aligned Segments"], loc="upper right")
plt.show()


In [ ]:
# Calculate audio durations minus interruptions
aligned_durations= [offset - onset for onset, offset in zip(audio_onsets_prelim, audio_offsets_prelim)]
interruption_durations = [end - start for start, end in interruptions_prelim]

# Adjust aligned durations by subtracting interruptions
for i in range(len(aligned_durations)):
    for start, end in interruptions_prelim:
        if audio_onsets_prelim[i] < end and audio_offsets_prelim[i] > start:  # Check overlap
            overlap = max(0, min(audio_offsets_prelim[i], end) - max(audio_onsets_prelim[i], start))
            aligned_durations[i] -= overlap

# Print the adjusted durations
for i in range(len(aligned_durations)):
    print(f"Stimulus {i+1} duration: {len(stimuli[i])/raw_sfreq:.2f} seconds, "
          f"Aligned - Interruption duration: {aligned_durations[i]/raw_sfreq:.2f} seconds")

Below we cross-correlate the audio playback with the stimuli split at the interruptions identified above for a precise audio alignment

In [ ]:
deviation_threshold = 2  # seconds


audio_onsets = []
audio_offsets = []
interruptions = []
part_starts = []
part_ends = []

for i, (onset, offset) in enumerate(zip(audio_onsets_prelim, audio_offsets_prelim)):
    overlapping_interruptions = [(int_start, int_end) for int_start, int_end in interruptions_prelim if not (int_end < onset or int_start > offset)]
    if not overlapping_interruptions:
        # cross-correlate the whole audio for the non-interrupted parts
        stim = stimuli[i]
        stim_rev = stim[::-1]
        # cross-correlating stimulus and audio channel, identifying onset
        correlation = signal.correlate(audio, stim, mode="full")
        lags = signal.correlation_lags(len(audio), len(stim), mode="full")
        assert correlation.shape[0] == lags.shape[0]
        if i == 0:  # part1_d3p6_wt.wav
            while lags[np.argmax(correlation)] > len(audio)/2:
                    correlation[np.argmax(correlation)] = -np.inf  # to avoid picking the wrong peak
            audio_onsets.append(lags[np.argmax(correlation)])  # manually adjust for part1_d3p6_wt.wav
        else:
            audio_onsets.append(lags[np.argmax(correlation)])
        # cross-correlating for offset
        correlation_rev = signal.correlate(audio[::-1], stim_rev, mode="full")
        lags_rev = signal.correlation_lags(len(audio[::-1]), len(stim_rev), mode="full")
        assert correlation_rev.shape[0] == lags_rev.shape[0]
        if i == 0:  # part1_d3p6_wt.wav
            while lags_rev[np.argmax(correlation_rev)] < len(audio)/2:
                correlation_rev[np.argmax(correlation_rev)] = -np.inf  # to avoid picking the wrong peak
            audio_offsets.append(len(audio) - (lags_rev[np.argmax(correlation_rev)]))  # manually adjust for part1_d3p6_wt.wav
        else:
            audio_offsets.append(len(audio) - lags_rev[np.argmax(correlation_rev)])
        print(f"Processed audio {i+1}: onset at {audio_onsets[-1]/raw_sfreq}, offset at {audio_offsets[-1]/raw_sfreq}")
        part_starts.append(audio_onsets[-1])
        part_ends.append(audio_offsets[-1])
        assert audio_offsets[-1] - audio_onsets[-1] == len(stim)
        assert audio[audio_onsets[-1]:audio_offsets[-1]].shape[0] == stim.shape[0]
    else:
        # cross-correlate only the non-interrupted parts
        stim = stimuli[i]
        stim_split = [0]  # indices to split the stimulus
        existing_interruption = 0
        for int_start, int_end in overlapping_interruptions:
            # cross-correlating stimulus and audio channel, identifying onset
            stim_split.append(int_start - onset - existing_interruption)
            existing_interruption += (int_end - int_start)
        stim_split.append(len(stim))
        actual_onsets_part = []
        actual_offsets_part = []
        for j, (part_onset, part_offset) in enumerate(zip(stim_split[:-1], stim_split[1:])):
            stim_part = stim[part_onset:part_offset]
            stim_part_len = len(stim_part)
            # cross-correlating stimulus part and audio channel, identifying onset
            correlation = signal.correlate(audio, stim_part, mode="full")
            lags = signal.correlation_lags(len(audio), len(stim_part), mode="full")
            assert correlation.shape[0] == lags.shape[0]
            if i == 0:  # part1_d3p6_wt.wav
                while lags[np.argmax(correlation)] > len(audio)/2:
                    correlation[np.argmax(correlation)] = -np.inf  # to avoid picking the wrong peak
            #while abs(lags[np.argmax(correlation)] - (onset + part_onset)) / raw_sfreq > deviation_threshold:
            #    correlation[np.argmax(correlation)] = -np.inf  # to avoid picking the wrong peak
            actual_onsets_part.append(lags[np.argmax(correlation)])
            # calculate the offset
            actual_offset = lags[np.argmax(correlation)] + stim_part_len
            actual_offsets_part.append(actual_offset)
        audio_onsets.append(actual_onsets_part[0])
        audio_offsets.append(actual_offsets_part[-1])
        for actual_start, actual_end in zip(actual_onsets_part[1:], actual_offsets_part[:-1]):
            interruptions.append((actual_end, actual_start))
        print(f"Processed audio {i+1}: onset at {audio_onsets[-1]/raw_sfreq}, offset at {audio_offsets[-1]/raw_sfreq}")
        assert sum([end - start for start, end in zip(actual_onsets_part, actual_offsets_part)]) == len(stim), f"Sum of parts duration does not match stimulus duration for audio {i+1}"
        part_starts.extend(actual_onsets_part)
        part_ends.extend(actual_offsets_part)

In [ ]:
# visualize interruptions and aligned segments
audio_s = np.arange(len(audio)) / raw_sfreq  # time axis in seconds
plt.figure(figsize=(10, 5))
plt.plot(audio_s, audio, label="Audio Sequence", alpha=0.7)

# Highlight parts between audio onsets and offsets
for onset, offset in zip(part_starts, part_ends):
    plt.axvspan(onset/raw_sfreq, offset/raw_sfreq, color='orange', alpha=0.3, label="Audio playback")

## Highlight interruptions
#for i, (start, end) in enumerate(interruptions):
#    plt.axvspan(start/raw_sfreq, end/raw_sfreq, color='red', linestyle='--', alpha=0.3, label = "Interruption")
#    plt.text(int(np.mean([start/raw_sfreq, end/raw_sfreq])), 0, f"Interruption {i}", color='black', fontsize=8, ha='center')

# Add labels and legend
plt.title("Interruptions (red, 0-indexed)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(["Audio Sequence", "Aligned Segments"], loc="upper right")
plt.show()


In [ ]:
# Calculate audio durations minus interruptions
aligned_durations= [offset - onset for onset, offset in zip(audio_onsets, audio_offsets)]
interruption_durations = [end - start for start, end in interruptions]

# Adjust aligned durations by subtracting interruptions
for i in range(len(aligned_durations)):
    for start, end in interruptions:
        if audio_onsets[i] < end and audio_offsets[i] > start:  # Check overlap
            overlap = max(0, min(audio_offsets[i], end) - max(audio_onsets[i], start))
            aligned_durations[i] -= overlap

# Print the adjusted durations
for i in range(len(aligned_durations)):
    print(f"Stimulus {i+1} duration: {len(stimuli[i])/raw_sfreq:.2f} seconds, "
          f"Aligned - Interruption duration: {aligned_durations[i]/raw_sfreq:.2f} seconds")

In [ ]:
# check audio alignment after cropping interruptions
audio_cropped = np.copy(audio_centered)
for start, end in interruptions:
    audio_cropped[start:end] = np.nan  # zero out interruptions
audio_cropped = audio_cropped[~np.isnan(audio_cropped)]  # remove NaNs
audio_onsets_cropped = []
audio_offsets_cropped = []
for onset, offset in zip(audio_onsets_prelim, audio_offsets_prelim):
    # find new onset
    onset_cropped = np.sum([1 for start, end in interruptions if end <= onset])  # number of interruptions before onset
    audio_onsets_cropped.append(onset - sum((end - start) for start, end in interruptions if end <= onset))
    # find new offset
    offset_cropped = np.sum([1 for start, end in interruptions if end <= offset])  # number of interruptions before offset
    audio_offsets_cropped.append(offset - sum((end - start) for start, end in interruptions if end <= offset))

# plotting for sanity check
for i in range(len(audios)):
    plt.figure()
    plt.title(f"Audio Segment from {audios[i]}")
    plt.plot(audio_cropped[audio_onsets_cropped[i]:audio_offsets_cropped[i]])
    plt.plot(stimuli[i], alpha=0.7)
    plt.xlabel("Samples")
    plt.ylabel("Amplitude")
    plt.show()

## mark bad segments

In [ ]:
# Create annotations for interruptions
bad_interruption_annotations = mne.Annotations(
    onset=[start / raw_sfreq for start, _ in interruptions],  # start times in seconds
    duration=[(end - start) / raw_sfreq for start, end in interruptions],  # durations in seconds
    description=['bad_interruption'] * len(interruptions)  # label as bad_interruption
)

# Add annotations to raw_bipol
raw_bipol.set_annotations(bad_interruption_annotations)

# Create annotations for bad_prepend and bad_append
bad_prepend_annotations = mne.Annotations(
    onset=[max(0, (start / raw_sfreq) - bad_prepend) for start, _ in interruptions],  # start times in seconds
    duration=[bad_prepend for start, _ in interruptions],  # durations in seconds
    description=['bad_prepend'] * len(interruptions)  # label as bad_prepend
)

bad_append_annotations = mne.Annotations(
    onset=[(end / raw_sfreq) for _, end in interruptions],  # start times in seconds
    duration=[bad_append] * len(interruptions),  # durations in seconds
    description=['bad_append'] * len(interruptions)  # label as bad_append
)

# Combine all annotations
raw_bipol.set_annotations(raw_bipol.annotations + bad_prepend_annotations + bad_append_annotations)

raw_bipol.plot(scalings='auto')

In [ ]:
# add audio annotations
audio_annotations = mne.Annotations(
    onset=[start / raw_sfreq for start in audio_onsets],
    duration=[(end - start) / raw_sfreq for start, end in zip(audio_onsets, audio_offsets)],
    description=['audio_aligned'] * len(audio_onsets)
)

raw_bipol.set_annotations(raw_bipol.annotations + audio_annotations)
raw_bipol.plot(scalings='auto')

# print parameters

In [ ]:
print(f"path = '/{subject}/iEEG/{subject}_KEC.mat'")
print(f"outpath = 'raw'")
print(f"audiopath = '../mono/'")
print(f"audio_channel = '{audio_channel}'")
print(f"audio_onsets = {[int(i) for i in audio_onsets]}")
print(f"audio_offsets = {[int(i) for i in audio_offsets]}")
try:
    print(f'interruptions = {[ (int(start), int(end)) for start, end in interruptions ]}')
except NameError:
    print("interruptions = []")
try:
    print(f"bads = {bads}")
except NameError:
    print("bads = []")
